# Training YOLO26n untuk Deteksi Buku, Rak (Shelf) & Pena — Kaggle Notebook

Notebook ini memakai **10 dataset asli yang sudah Anda kumpulkan/export dari Roboflow**
(bukan lagi dataset publik generik yang saya rekomendasikan sebelumnya) — sudah saya periksa
satu per satu (isi data.yaml, jumlah gambar, format label bbox vs poligon, provenance
Roboflow) dan diverifikasi lokal: setelah digabung, **7.931 gambar, 120.302 objek berlabel,
0 baris rusak, 0 kesalahan index kelas**. Lihat Bagian 3 untuk rincian tiap file.

Arsitektur & langkah training identik dengan `train_on_kaggle.ipynb` (domain obat) — hanya
dataset dan skema kelasnya yang berbeda (kini 3 kelas: `book`, `shelf`, `pen`).

## Cara pakai notebook ini di Kaggle
1. Buka https://www.kaggle.com/code -> **New Notebook**, lalu paste/upload file ini.
2. **Settings -> Accelerator -> GPU T4 x2** (atau P100).
3. **Settings -> Internet -> On**.
4. Upload KESEMBILAN file di Bagian 3 (bukan yang ke-10, lihat catatan pengecualian) sebagai
   Kaggle Dataset (`+ Add Data -> Upload`), lalu `+ Add Input` ke notebook ini.
5. **Run All**.
6. Download `book_model_export.zip` dari panel Output Kaggle.

## 0. Level 0 dulu — cek apakah Anda BUTUH training sama sekali

Model COCO-pretrained yang sudah ada di proyek ini (`models/yolo26n.onnx`) **sudah punya
kelas "book"** bawaan (lihat `src/pipeline.py`, `COCO_CLASSES`). Sebelum menghabiskan kuota
GPU, coba dulu tanpa training apa pun:
```bash
python scripts/test_webcam.py --config config/book_pipeline_config.yaml
```
Notebook ini (Level 1) baru diperlukan kalau akurasi Level 0 kurang untuk kondisi rak Anda,
atau kalau Anda ingin kelas tambahan yang TIDAK ada di COCO: **"shelf" (rak) dan "pen"
(pena)** — dua kelas ini hanya bisa didapat lewat training kustom di bawah.

## 1. Cek GPU aktif

In [ ]:
!nvidia-smi

## 2. Install Ultralytics

In [ ]:
!pip install -q ultralytics onnx onnxslim
import ultralytics
ultralytics.checks()

## 3. Dataset — 10 file yang Anda upload, sudah diperiksa satu per satu

Setiap file diekspor Roboflow dengan nama kelas yang TIDAK selalu bersih (beberapa berisi
judul proyek/tanggal, bukan nama kelas asli) — sudah dicek isi `data.yaml`, provenance
(`roboflow.workspace`/`project`), dan **distribusi pemakaian tiap index kelas di label
asli** (bukan cuma daftar nama kelasnya) untuk memastikan artinya benar sebelum digabung:

| # | File | Proyek Roboflow asli | Kelas asli (index: nama) | Objek terpakai | Dipetakan ke |
|---|---|---|---|---|---|
| 1 | `book.v1i.yolo26(1).zip` | zebra-learn/book-4abtl v1 | 0: book | 2.384 | `book` |
| 2 | `book.v1i.yolo26.zip` | **saca-setya-wardana**/book-omfeo-r6slr v1 | 0: book | 1.881 (poligon) | `book` |
| 3 | `book.v2i.yolo26(1).zip` | yrden/book-zbbr0 v2 | 0: "book_image - v6..." (nama proyek, BUKAN kelas asli) | 52.095 | `book` |
| 4 | `book.v2i.yolo26.zip` | library-rjncz/book-6wkuh v2 | 0 & 1: nama korup ("---...", "book segmentation - v5...") | 36.101 (poligon) | `book` (keduanya) |
| 5 | `Book spline detection.v1i.yolov8.zip` | books-26cz6/book-spline-detection v1 | 0: "Book spine..." (dominan), 1: "object" (129x, noise) | 22.852 (poligon) | `book` (keduanya) |
| 6 | `Book_Pen_Model.v1i.yolo26.zip` | working-vce8g/book-pen-model v1 | 0: Book, 1: Pen (BERSIH) | 2.707 + 380 | `book` + `pen` |
| 7 | `book.v2-roboflow-instant-1--eval-.yolo26.zip` | sadang/book-bzuxs v2 | 0: book (3x), 1: books (808x) | 811 | `book` (keduanya) |
| 8 | `shelf.v3i.yolo26.zip` | university-of-science-vnu-zlz2c/shelf-tzfro v3 | 0: shelf (BERSIH) | 518 | `shelf` |
| 9 | `shelf.v2-shelfv5.yolo26.zip` | robocup2022-kogzd/shelf-6pwdx v2 | 0: shelf (BERSIH) | 573 | `shelf` |
| 10 | `BOOK.v1i.yolo26.zip` | blueworkspace/book-c8llt v1 | 0: "-" (2x), **1: "1dcode" (890x, DOMINAN)** | 892 | **DIKECUALIKAN** |

**Semua 10 dataset berlisensi CC BY 4.0** (atribusi wajib bila dipakai untuk model akhir —
lihat daftar lengkap workspace/project di tabel, sudah cukup untuk kutipan).

**Kenapa file #10 dikecualikan:** meski dinamai "BOOK.v1i", isi labelnya didominasi kelas
`1dcode` (kode batang 1 dimensi/barcode di sampul buku), bukan objek "buku" itu sendiri —
kalau ikut digabung sebagai `book`, model akan salah belajar (bentuk barcode dianggap bentuk
buku). **Ini BUKAN dibuang begitu saja** — file ini relevan untuk pekerjaan lain: tahap
"Verification" di visi NSTC-Mackay (OCR + **barcode GS1**) yang saat ini masih "belum ada" di
proyek ini (lihat laporan `.docx`, Bagian 6). Simpan file ini terpisah untuk R&D barcode nanti.

**Kenapa file #3 dan #4 kelas aslinya "korup":** ini pola umum Roboflow — kalau sebuah
project cuma py 1 kelas dan pernah di-rename/re-export, nama kelas kadang tertimpa jadi judul
proyek+tanggal export, atau (kasus #4) placeholder `---` untuk kelas versi lama yang sudah
tidak dipakai tapi datanya masih ada. Sudah dicek langsung distribusi pemakaian index (bukan
cuma nama), jadi pemetaan ke `book` di sel di bawah aman dilakukan.

In [ ]:
import glob, pathlib, shutil, yaml, zipfile, hashlib

SPLITS = ("train", "valid", "test")

# Pemetaan kelas per dataset -- kunci oleh (roboflow.workspace, roboflow.project) yang
# tertulis DI DALAM data.yaml tiap zip, bukan oleh nama file (nama file bisa berubah kalau
# Anda unggah ulang / rename). Ini hasil pemeriksaan manual pada isi label asli tiap dataset
# (distribusi pemakaian index, dicek nyata, bukan asumsi dari nama kelas) -- lihat Bagian 3.
#
# value = {local_class_index: "nama_kelas_gabungan"} ; index yang TIDAK ada di dict = dibuang
# (dianggap di luar cakupan, bukan error).
REMAP = {
    ("zebra-learn", "book-4abtl"):                 {0: "book"},                    # book.v1i (1)
    ("saca-setya-wardana", "book-omfeo-r6slr"):    {0: "book"},                    # book.v1i (punya Anda sendiri!)
    ("yrden", "book-zbbr0"):                       {0: "book"},                    # book.v2i (1)
    ("library-rjncz", "book-6wkuh"):               {0: "book", 1: "book"},         # book.v2i
    ("books-26cz6", "book-spline-detection"):      {0: "book", 1: "book"},         # Book spline detection
    ("working-vce8g", "book-pen-model"):           {0: "book", 1: "pen"},          # Book_Pen_Model
    ("sadang", "book-bzuxs"):                      {0: "book", 1: "book"},         # book.v2-instant-eval
    ("university-of-science-vnu-zlz2c", "shelf-tzfro"): {0: "shelf"},              # shelf.v3i
    ("robocup2022-kogzd", "shelf-6pwdx"):          {0: "shelf"},                   # shelf.v2-shelfv5
    # ("blueworkspace", "book-c8llt") sengaja TIDAK didaftarkan -> dataset "1dcode"/barcode,
    # dikecualikan (lihat Bagian 3). Kalau ter-upload, sel ini akan melewatinya otomatis
    # dengan pesan jelas, BUKAN ikut tergabung diam-diam.
}
MERGED_NAMES = ["book", "shelf", "pen"]
name_to_idx = {n: i for i, n in enumerate(MERGED_NAMES)}


def short_name(dname, stem, ext=""):
    # Beberapa export Roboflow ("instant eval") memberi nama file berupa key S3
    # ter-base64 yang sangat panjang (150+ karakter) -- bisa bikin error "File name too
    # long" di sebagian filesystem. Dipendekkan pakai hash, deterministik per file.
    h = hashlib.sha1(stem.encode()).hexdigest()[:12]
    return f"{dname}__{h}{ext}"


def convert_label_line(line):
    parts = line.strip().split()
    if not parts:
        return None
    cls, coords = parts[0], [float(x) for x in parts[1:]]
    if len(coords) == 4:
        return cls, coords
    if len(coords) < 6 or len(coords) % 2 != 0:
        return None
    xs, ys = coords[0::2], coords[1::2]
    x_min, x_max, y_min, y_max = min(xs), max(xs), min(ys), max(ys)
    return cls, [(x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min]


def find_data_yaml_roots(base):
    return {pathlib.Path(p).parent for p in glob.glob(f"{base}/**/data.yaml", recursive=True)}


raw_roots = find_data_yaml_roots("/kaggle/input")
already_covered_dirs = set(raw_roots)
zips = [
    zp for zp in glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if pathlib.Path(zp).parent not in already_covered_dirs
]
if zips:
    print(f"Ditemukan {len(zips)} file .zip mentah di /kaggle/input/, mengekstrak dulu...")
    for zp in zips:
        zp = pathlib.Path(zp)
        dest = pathlib.Path("/kaggle/working") / f"{zp.parent.name}__{zp.stem}_raw"
        if not dest.exists():
            print(f"  [EKSTRAK] {zp} -> {dest}")
            with zipfile.ZipFile(zp) as zf:
                zf.extractall(dest)
        raw_roots |= find_data_yaml_roots(str(dest))

if not raw_roots:
    print("=== Isi /kaggle/input/ saat ini (untuk diagnosa) ===")
    for p in sorted(glob.glob("/kaggle/input/*")):
        print(" ", p)
    raise FileNotFoundError(
        "Tidak ditemukan data.yaml/.zip di /kaggle/input/. Upload ke Kaggle Datasets TIDAK "
        "otomatis menyambungkannya ke notebook ini -- buka '+ Add Input' -> tambahkan dataset "
        "yang sudah Anda upload -> jalankan ulang sel ini."
    )

raw_roots = sorted(raw_roots)
print(f"Ditemukan {len(raw_roots)} dataset ter-attach.\n")

MERGED_ROOT = pathlib.Path("/kaggle/working/book_shelf_pen_merged")
if MERGED_ROOT.exists():
    shutil.rmtree(MERGED_ROOT)

totals = {"images": 0, "bbox": 0, "poly": 0, "bad": 0, "dropped_off_scope": 0}
per_class_totals = {}
skipped_unmapped = []

for raw_root in raw_roots:
    cfg = yaml.safe_load(open(raw_root / "data.yaml"))
    rf = cfg.get("roboflow", {})
    key = (rf.get("workspace"), rf.get("project"))
    remap = REMAP.get(key)
    dname = f"{key[0]}_{key[1]}" if key[0] else raw_root.name
    if remap is None:
        skipped_unmapped.append((raw_root.name, key, cfg.get("names")))
        print(f"[LEWATI] {raw_root.name} (roboflow={key}): tidak ada di tabel REMAP -- kemungkinan "
              f"dataset barcode/1dcode (lihat Bagian 3) atau dataset BARU yang belum diperiksa. "
              f"Kelas aslinya: {cfg.get('names')} -- beri tahu saya nama kelas yang benar kalau ini "
              f"seharusnya ikut digabung, supaya bisa ditambahkan ke REMAP.")
        continue

    for split in SPLITS:
        src_img, src_lbl = raw_root / split / "images", raw_root / split / "labels"
        if not src_img.exists():
            continue
        dst_img, dst_lbl = MERGED_ROOT / split / "images", MERGED_ROOT / split / "labels"
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)
        for p in src_img.iterdir():
            if p.is_file():
                shutil.copy(p, dst_img / short_name(dname, p.stem, p.suffix))
                totals["images"] += 1
        for lbl_path in src_lbl.glob("*.txt"):
            out_lines = []
            for line in lbl_path.read_text().splitlines():
                if not line.strip():
                    continue
                n_tok = len(line.strip().split()) - 1
                result = convert_label_line(line)
                if result is None:
                    totals["bad"] += 1
                    continue
                local_cls, (cx, cy, w, h) = result
                canonical = remap.get(int(local_cls))
                if canonical is None:
                    totals["dropped_off_scope"] += 1
                    continue
                global_cls = name_to_idx[canonical]
                out_lines.append(f"{global_cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
                per_class_totals[canonical] = per_class_totals.get(canonical, 0) + 1
                if n_tok == 4:
                    totals["bbox"] += 1
                else:
                    totals["poly"] += 1
            out_stem = short_name(dname, lbl_path.stem)
            (dst_lbl / f"{out_stem}.txt").write_text("\n".join(out_lines) + ("\n" if out_lines else ""))
    print(f"[OK] {dname}: digabung ({remap})")

yaml.safe_dump(
    {"path": str(MERGED_ROOT), "train": "train/images", "val": "valid/images", "test": "test/images",
     "nc": len(MERGED_NAMES), "names": MERGED_NAMES},
    open(MERGED_ROOT / "data.yaml", "w"), sort_keys=False,
)

print(f"\nTOTAL gambar: {totals['images']}")
print(f"TOTAL objek: bbox_asli={totals['bbox']}  poligon_dikonversi={totals['poly']}  "
      f"baris_rusak={totals['bad']}  di_luar_cakupan_dibuang={totals['dropped_off_scope']}")
print(f"Per kelas gabungan: {per_class_totals}")
print(f"Dataset gabungan siap di: {MERGED_ROOT}")
if skipped_unmapped:
    print(f"\n{len(skipped_unmapped)} dataset ter-attach TIDAK ikut digabung (lihat pesan [LEWATI] di atas).")

## 4. `data.yaml` sudah otomatis ditulis oleh sel di atas

Sel di atas menulis `data.yaml` gabungan ke `/kaggle/working/book_shelf_pen_merged/data.yaml`
(3 kelas: book, shelf, pen). **Lewati sel di bawah** kecuali Anda menambah dataset lain di
luar 9 yang sudah dipetakan.

In [ ]:
data_yaml_content = '''
path: /kaggle/input/NAMA_DATASET_BARU_ANDA
train: images/train
val: images/val

names:
  0: book
  1: shelf
  2: pen
'''

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(data_yaml_content)

print(data_yaml_content)

## 5. Training

Identik dengan notebook obat: `yolo26n.pt` (nano, COCO-pretrained), `epochs=120`,
`patience=30`. Dataset gabungan (~7.900 gambar, 3 kelas) jauh lebih kecil dari dataset obat
(52 kelas), jadi training biasanya konvergen dalam waktu yang wajar di T4 (perkirakan
1-2 jam, tergantung distribusi kelas -- kelas `pen` jauh lebih sedikit contohnya, lihat catatan
Bagian 6).

In [ ]:
try:
    from ultralytics import YOLO
except ModuleNotFoundError:
    print("ultralytics belum terinstall di sesi ini, menginstall dulu...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "onnx", "onnxslim"], check=True)
    from ultralytics import YOLO

import pathlib
_merged = pathlib.Path("/kaggle/working/book_shelf_pen_merged/data.yaml")
DATA_YAML = str(_merged) if _merged.exists() else "/kaggle/working/data.yaml"
print("Training dengan:", DATA_YAML)

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATA_YAML,
    epochs=120,
    imgsz=640,
    batch=16,
    device=0,
    patience=30,
    workers=4,
    project="/kaggle/working/runs",
    name="book_yolo26n",
)

## 6. Validasi hasil

**Catatan realistis soal ketidakseimbangan kelas:** dari 120.302 objek gabungan, `book`
mendominasi (118.831, 98,8%), `shelf` jauh lebih sedikit (1.091), dan `pen` paling sedikit
(380). Wajar kalau mAP kelas `pen` lebih rendah dari `book` -- kalau ternyata sangat rendah
dan `pen` tidak penting untuk kebutuhan Anda, pertimbangkan melatih ulang tanpa kelas ini
(hapus `working-vce8g/book-pen-model` dari upload) daripada memaksakan hasil yang lemah.

In [ ]:
metrics = model.val()
print(metrics.box.maps)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

## 7. Export ke ONNX (dipakai `src/detector.py` di laptop, tidak berubah)

In [ ]:
best_weights = "/kaggle/working/runs/book_yolo26n/weights/best.pt"
best_model = YOLO(best_weights)
onnx_path = best_model.export(format="onnx", imgsz=640, opset=12, simplify=True)
print("ONNX tersimpan di:", onnx_path)

## 8. Bungkus hasil untuk di-download ke laptop

```bash
unzip book_model_export.zip -d ~/Downloads/book_model_export
cp ~/Downloads/book_model_export/best.onnx  medicine_shelf_vision/models/book_yolo26n.onnx
cp ~/Downloads/book_model_export/classes.txt medicine_shelf_vision/data/book_classes.txt

python scripts/test_webcam.py --config config/book_pipeline_config.yaml \\
    --onnx models/book_yolo26n.onnx --classes data/book_classes.txt

python scripts/run_shelf_search.py --config config/book_pipeline_config.yaml \\
    --target <id_buku> --live --image foto_rak_buku.jpg
```

`data/book_classes.txt` akan berisi 3 baris: `book`, `shelf`, `pen` (urutan sesuai
`model.names` hasil training, otomatis konsisten dengan index di ONNX).

In [ ]:
import shutil, os

export_dir = "/kaggle/working/book_model_export"
os.makedirs(export_dir, exist_ok=True)

shutil.copy(onnx_path, os.path.join(export_dir, "best.onnx"))
shutil.copy(best_weights, os.path.join(export_dir, "best.pt"))

with open(os.path.join(export_dir, "classes.txt"), "w") as f:
    for idx, name in model.names.items():
        f.write(name + "\n")

shutil.make_archive("/kaggle/working/book_model_export", "zip", export_dir)
print("Selesai. Download book_model_export.zip dari panel Output Kaggle.")